In [1]:
from cube_nn import CubeValueResNet
from training import train_on_value_dataset
from torch import optim
import matplotlib.pyplot as plt
import torch
from cube_nn import NNValueFunctionType
from cube_datasets import TrainingValueDataset

device = "cuda"

In [2]:
# name = 'resnet2.2'
name = 'resnet2'

# Load a previously saved net
I_NET = 7000
net = CubeValueResNet()
net.load_state_dict(torch.load(f'temp_models/{name}/cube_value_resnet_iter_{I_NET}.pth'))

/tmp/ipykernel_2012333/88339321.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load(f'temp_models/{name}/cube_value_resnet_iter_{I_NET}.pth'))

<All keys matched successfully>

In [3]:
N_ROLLOUTS = 1000
N_MOVES_MAX = 30

dataset = TrainingValueDataset.create_from_trajectories(
        net.get_cube_to_tensor(),
        N_ROLLOUTS, 
        N_MOVES_MAX, 
        device="cpu",
    )

Generating trajectories:   0%|          | 0/1000 [00:00<?, ?it/s]

In [4]:
# Estimate values for all cubes in the dataset
import numpy as np

net.eval()
net.to(device)

# Group cubes by number of moves
cubes_by_moves = {}
for i in range(N_MOVES_MAX + 1):
    cubes_by_moves[i] = []

# Organize cubes by their number of moves
for cube_tensor, value in dataset:
    cubes_by_moves[value.item()].append(cube_tensor)

# Estimate values for each group
move_numbers = []
avg_values = []
std_values = []
min_values = []
max_values = []

with torch.no_grad():
    for moves in sorted(cubes_by_moves.keys()):
        if len(cubes_by_moves[moves]) > 0:
            # Stack all cubes for this move count
            cube_batch = torch.stack(cubes_by_moves[moves]).to(device)
            
            # Get predictions
            predictions = net(cube_batch).cpu().numpy().flatten()
            
            # Calculate statistics
            move_numbers.append(moves)
            avg_values.append(np.mean(predictions))
            std_values.append(np.std(predictions))
            min_values.append(np.min(predictions))
            max_values.append(np.max(predictions))

print(f"Processed {len(move_numbers)} different move counts")
print(f"Total cubes analyzed: {sum(len(cubes_by_moves[m]) for m in cubes_by_moves)}")

Processed 30 different move counts
Total cubes analyzed: 30000


In [5]:
# Combined plot with all statistics
plt.figure(figsize=(12, 6))

plt.plot(move_numbers, avg_values, marker='o', linewidth=2, markersize=6, label='Average', color='blue')
plt.fill_between(move_numbers, 
                 [avg - std for avg, std in zip(avg_values, std_values)],
                 [avg + std for avg, std in zip(avg_values, std_values)],
                 alpha=0.3, color='blue', label='±1 Std Dev')
plt.plot(move_numbers, min_values, marker='s', linewidth=1.5, markersize=5, label='Min', color='green', linestyle='--')
plt.plot(move_numbers, max_values, marker='^', linewidth=1.5, markersize=5, label='Max', color='red', linestyle='--')

plt.xlabel('Number of Moves', fontsize=12)
plt.ylabel('Predicted Value', fontsize=12)
plt.title('Network Value Predictions by Number of Moves', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save the figure
plt.savefig(f'figs/{name}/prediction_analysis_{I_NET}.png', dpi=300, bbox_inches='tight')

plt.show()


In [9]:
# Find cubes with actual value 4 that the model predicts as ~3
net.eval()
net.to(device)


N_MOVES = 5
VALUE_THRESHOLD = 4.5

value_cubes = []
value_predictions = []

with torch.no_grad():
    for cube_tensor, value in dataset:
        if value.item() == N_MOVES:
            cube_batch = cube_tensor.unsqueeze(0).to(device)
            prediction = net(cube_batch).cpu().item()
            value_cubes.append(cube_tensor)
            value_predictions.append(prediction)
# Find cubes predicted as close to 3 (let's say < 3.5)
misclassified = [(i, pred) for i, pred in enumerate(value_predictions) if pred < VALUE_THRESHOLD]

print(f"Total cubes with actual value {N_MOVES}: {len(value_cubes)}")
print(f"Cubes with value {N_MOVES} predicted as < {VALUE_THRESHOLD}: {len(misclassified)}")
print(f"\nPredictions for value-{N_MOVES} cubes predicted as < {VALUE_THRESHOLD}:")
for idx, pred in misclassified:
    print(f"  Cube {idx}: predicted value = {pred:.4f}")

Total cubes with actual value 5: 1000
Cubes with value 5 predicted as < 4.5: 42

Predictions for value-5 cubes predicted as < 4.5:
  Cube 16: predicted value = 4.3841
  Cube 22: predicted value = 4.4028
  Cube 25: predicted value = 4.1086
  Cube 117: predicted value = 4.2796
  Cube 151: predicted value = 4.0463
  Cube 152: predicted value = 4.2084
  Cube 154: predicted value = 4.4810
  Cube 159: predicted value = 3.9507
  Cube 178: predicted value = 4.4216
  Cube 229: predicted value = 4.3856
  Cube 241: predicted value = 4.2906
  Cube 265: predicted value = 4.3123
  Cube 272: predicted value = 4.2937
  Cube 273: predicted value = 4.0222
  Cube 298: predicted value = 4.1745
  Cube 309: predicted value = 4.0811
  Cube 320: predicted value = 4.2618
  Cube 354: predicted value = 4.3457
  Cube 372: predicted value = 3.9659
  Cube 397: predicted value = 4.3649
  Cube 407: predicted value = 4.4449
  Cube 408: predicted value = 4.2514
  Cube 431: predicted value = 4.3164
  Cube 487: predicted

In [11]:
# Print out the actual cube states that are misclassified
from cube import Cube
from solvers import BFSSolverDataset

solver = BFSSolverDataset(dataset_moves=N_MOVES, max_bfs_depth=N_MOVES)

print(f"\n{'='*60}")
print(f"CUBE STATES WITH VALUE {N_MOVES} PREDICTED AS < {VALUE_THRESHOLD}:")
print(f"{'='*60}\n")

for idx, pred in misclassified: 
    print(f"\nCube {idx} - Predicted: {pred:.4f}, Actual: {N_MOVES}")
    print("-" * 60)
    
    # Convert tensor back to cube
    # The tensor is one-hot encoded with shape (54, 6) or (324,) if flattened
    cube_tensor = value_cubes[idx]
    
    # If one-hot encoded, get the argmax to recover the original values
    if len(cube_tensor.shape) == 2:  # shape (54, 6)
        cube_state = cube_tensor.argmax(dim=1).reshape(6, 3, 3).numpy().astype(np.uint8)
    else:  # shape (324,) - one-hot flattened
        cube_state = cube_tensor.reshape(54, 6).argmax(dim=1).reshape(6, 3, 3).numpy().astype(np.uint8)
    
    # Create cube object from the state
    cube = Cube(cube=cube_state)

    # Solve to verify true value
    solution = solver(cube)

    print(f"Solution: {solution}")
    print(f"Solution length: {len(solution)}")
    
    # Plot the cube
    # cube.plot()

Loaded full optimal value dataset from datasets/full/full_5.pt.

CUBE STATES WITH VALUE 5 PREDICTED AS < 4.5:


Cube 16 - Predicted: 4.3841, Actual: 5
------------------------------------------------------------
Solution: [F, D, F', B, L]
Solution length: 5

Cube 22 - Predicted: 4.4028, Actual: 5
------------------------------------------------------------
Solution: [U', B2, U, B', L2]
Solution length: 5

Cube 25 - Predicted: 4.1086, Actual: 5
------------------------------------------------------------
Solution: [D, L2, R2, D2, R']
Solution length: 5

Cube 117 - Predicted: 4.2796, Actual: 5
------------------------------------------------------------
Solution: [U, L, B, L2, B2]
Solution length: 5

Cube 151 - Predicted: 4.0463, Actual: 5
------------------------------------------------------------
Solution: [R', U2, R2, B2, L2]
Solution length: 5

Cube 152 - Predicted: 4.2084, Actual: 5
------------------------------------------------------------
Solution: [L, U', L2, U2, D2]
Solution 

In [8]:
# Let's try to find what scramble sequence leads to this cube state
# by testing different seeds with 2 moves

if len(misclassified) > 0:
    # Get the first misclassified cube
    idx, pred = misclassified[0]
    cube_tensor = value_2_cubes[idx]
    
    # Convert to cube state
    if len(cube_tensor.shape) == 2:
        target_state = cube_tensor.argmax(dim=1).reshape(6, 3, 3).numpy().astype(np.uint8)
    else:
        target_state = cube_tensor.reshape(54, 6).argmax(dim=1).reshape(6, 3, 3).numpy().astype(np.uint8)
    
    target_cube = Cube(cube=target_state)
    
    print("Searching for a scramble seed that produces this cube state...")
    print(f"Target cube predicted value: {pred:.4f}")
    print()
    
    # Search through seeds
    found = False
    for seed in range(10000):
        test_cube = Cube(n_scramble_moves=2, scramble_seed=seed)
        if test_cube == target_cube:
            print(f"Found matching seed: {seed}")
            
            # Now let's see what scramble sequence was generated
            test_cube2 = Cube()
            moves = test_cube2.scramble(2, seed)
            print(f"Scramble sequence: {moves}")
            print()
            
            # Let's check if this is actually just 1 move
            test_cube3 = Cube()
            test_cube3.move(moves)
            print(f"Cube after moves is solved: {test_cube3.is_solved()}")
            
            # Try applying just the first move
            test_cube4 = Cube()
            test_cube4.move(moves[0])
            print(f"Cube after ONLY first move equals target: {test_cube4 == target_cube}")
            
            # Try applying just the second move  
            test_cube5 = Cube()
            test_cube5.move(moves[1])
            print(f"Cube after ONLY second move equals target: {test_cube5 == target_cube}")
            
            found = True
            break
    
    if not found:
        print("No seed found in range 0-9999. The cube might be from a different scramble method.")

NameError: name 'value_2_cubes' is not defined

In [ ]:
test_cube = Cube(n_scramble_moves=2, scramble_seed=1159)

In [ ]:
# Let's analyze the scramble method to find the bug
from cube import moves, Face, MoveSpecifier, Move

def detailed_scramble_analysis(seed, n_moves=2):
    """Trace through the scramble logic step by step"""
    
    print(f"Analyzing scramble with seed={seed}, n_moves={n_moves}")
    print("="*60)
    
    # Replicate the scramble logic
    rng = np.random.default_rng(seed)
    faces_list = list(Face)
    
    # Generate initial moves
    initial_moves = [Move(faces_list[rng.integers(0, len(faces_list))], 
                         MoveSpecifier(rng.integers(1, 4))) for _ in range(n_moves)]
    
    print(f"Initial moves: {initial_moves}")
    
    # Now simulate the filtering logic
    move_list = initial_moves.copy()
    i = 0
    iteration = 0
    
    while i < len(move_list) - 1:
        iteration += 1
        print(f"\nIteration {iteration}, i={i}, moves={move_list}")
        
        # Check condition 1: consecutive moves on same face
        if move_list[i].face == move_list[i + 1].face:
            print(f"  -> Found consecutive same face at positions {i} and {i+1}")
            removed = move_list.pop(i + 1)
            new_move = Move(faces_list[rng.integers(0, len(faces_list))], 
                           MoveSpecifier(rng.integers(1, 4)))
            move_list.append(new_move)
            print(f"  -> Removed {removed}, appended {new_move}")
            i = max(0, i - 1)
            continue
            
        # Check condition 2: ABA pattern with opposite faces
        if i < len(move_list) - 2:
            if (move_list[i].face == move_list[i + 2].face and 
                move_list[i].face == OPPOSITE_FACES.get(move_list[i + 1].face)):
                print(f"  -> Found ABA pattern at positions {i}, {i+1}, {i+2}")
                print(f"     {move_list[i].face.value} - {move_list[i+1].face.value} - {move_list[i+2].face.value}")
                removed = move_list.pop(i + 2)
                new_move = Move(faces_list[rng.integers(0, len(faces_list))], 
                               MoveSpecifier(rng.integers(1, 4)))
                move_list.append(new_move)
                print(f"  -> Removed {removed}, appended {new_move}")
                i -= 1
                continue
        
        i += 1
        
        if iteration > 20:
            print("\nToo many iterations, breaking...")
            break
    
    print(f"\nFinal moves: {move_list}")
    
    # Test if these moves are equivalent to a single move
    cube = Cube()
    cube.move(move_list)
    
    for single_move in moves:
        test_cube = Cube()
        test_cube.move(single_move)
        if test_cube == cube:
            print(f"\n*** BUG FOUND: Move sequence {move_list} is equivalent to single move {single_move} ***")
            return True
    
    print(f"\nNo bug found - cube is properly scrambled")
    return False

# Test a few seeds
for seed in range(20):
    if detailed_scramble_analysis(seed):
        break
    print("\n" + "-"*60 + "\n")

Analyzing scramble with seed=0, n_moves=2
Initial moves: [R', D]

Iteration 1, i=0, moves=[R', D]

Final moves: [R', D]

No bug found - cube is properly scrambled

------------------------------------------------------------

Analyzing scramble with seed=1, n_moves=2
Initial moves: [U', L2]

Iteration 1, i=0, moves=[U', L2]

Final moves: [U', L2]

No bug found - cube is properly scrambled

------------------------------------------------------------

Analyzing scramble with seed=2, n_moves=2
Initial moves: [R, F]

Iteration 1, i=0, moves=[R, F]

Final moves: [R, F]

No bug found - cube is properly scrambled

------------------------------------------------------------

Analyzing scramble with seed=3, n_moves=2
Initial moves: [L, B]

Iteration 1, i=0, moves=[L, B]

Final moves: [L, B]

No bug found - cube is properly scrambled

------------------------------------------------------------

Analyzing scramble with seed=4, n_moves=2
Initial moves: [L2, R']

Iteration 1, i=0, moves=[L2, R']

In [ ]:
# Import OPPOSITE_FACES for the analysis
from cube import OPPOSITE_FACES